In [ ]:
text = "Who is Isaac Newton?"
ExecutionProvider = "RyzenAILightExecutionProvider"
model_folder = "./model"

In [ ]:
import os

# Reference: https://github.com/microsoft/onnxruntime-genai/blob/c5af3edd45d25c75547118e15192dd1c6ffe7678/src/ryzenai/interface.cpp
# from winml import register_execution_providers_to_onnxruntime_genai
from winml import _get_ep_paths

os.environ["RYZENAI_EP_PATH"] = _get_ep_paths(ExecutionProvider)[ExecutionProvider]

In [ ]:
import json
import time

import onnxruntime_genai as og

# Load the base model and tokenizer
model = og.Model(model_folder)
tokenizer = og.Tokenizer(model)
tokenizer_stream = tokenizer.create_stream()

# Set the max length to something sensible by default,
# since otherwise it will be set to the entire context length
search_options = {}
search_options["max_length"] = 200

with open(os.path.join(model_folder, "chat_template.jinja"), "r", encoding="utf-8") as f:
    template_str = f.read()

# Apply the chat template via onnxruntime_genai (messages must be a JSON string)
messages = json.dumps([{"role": "user", "content": text}])
prompt = tokenizer.apply_chat_template(
    messages,
    template_str=template_str,
    add_generation_prompt=True,
)

# Encode the prompt using the tokenizer
input_tokens = tokenizer.encode(prompt)

# Create params and generator
params = og.GeneratorParams(model)
params.set_search_options(**search_options)
generator = og.Generator(model, params)

# Append input tokens to the generator
generator.append_tokens(input_tokens)

print("")
print("Output: ", end="", flush=True)

token_times = []

# Stream the output
while True:
    start_time = time.time()
    if generator.is_done():
        break
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    end_time = time.time()

    # Record the time for this token generation
    token_time = end_time - start_time
    token_times.append(token_time)

    print(tokenizer_stream.decode(new_token), end="", flush=True)

print()

# Calculate and display timing statistics
if token_times:
    total_tokens = len(token_times)
    avg_time = sum(token_times) / total_tokens

    print(f"Total tokens generated: {total_tokens}")
    print(f"Average time per token: {avg_time:.4f} seconds")
    print(f"Tokens per second: {total_tokens / sum(token_times):.2f}")

del generator